# 05 · EDA: heat treatment, lab effect and why rows aren't independent (EV)

Reproduces every number in the EDA briefing from the cleaned 17-4PH data. Run top to bottom; figures save to `figures/eda/` on your laptop (not committed: the repo is public).

- Cleaning steps: `docs/cleaning-log.md`
- What the statistics mean (Spearman, Kruskal–Wallis ε², grouped cross-validation): `docs/statistics-explained.md`
- Results written up: `docs/findings.md`

In [ ]:
import sys; sys.path.append('../../src')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from group5_clean import load_clean
from group5_stats import epsilon_squared, shuffle_threshold, within_treatment, spearman_per_setting, cross_validate
FIG = Path('../../figures/eda'); FIG.mkdir(parents=True, exist_ok=True)
ph = load_clean(verbose=True)

## 1. Named heat treatments
“Heat treated” covers very different recipes, so every comparison below uses the named treatment (`ht_route`).

In [ ]:
tab = (ph.dropna(subset=['YS_MPa']).groupby('ht_route')
       .agg(rows=('YS_MPa', 'size'), papers=('paper', 'nunique'), median_YS=('YS_MPa', 'median'),
            min_YS=('YS_MPa', 'min'), max_YS=('YS_MPa', 'max')).sort_values('rows', ascending=False))
tab

In [ ]:
order = ['As-printed', 'Solution only', 'Direct age H900', 'Solution + H900', 'Solution + H1025']
wrought_min = {'Solution + H900': 1170, 'Solution + H1025': 1000}   # ASTM A564 / AMS 5643
y = ph.dropna(subset=['YS_MPa'])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot([y[y.ht_route == r].YS_MPa for r in order], showfliers=False)
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels([r.replace(' + ', ' +\n') for r in order])
for i, r in enumerate(order, 1):
    v = y[y.ht_route == r].YS_MPa
    ax.scatter(np.random.normal(i, .06, len(v)), v, s=14, alpha=.7)
    if r in wrought_min: ax.hlines(wrought_min[r], i - .35, i + .35, linestyles='--', color='k')
ax.set_ylabel('Yield strength (MPa)'); ax.set_title('Yield by named heat treatment (dashed = wrought minimum)')
fig.tight_layout(); fig.savefig(FIG / 'yield_by_named_treatment.png', dpi=200)

## 2. Within one treatment, do the laser settings matter?
Spearman ρ, one point per paper × setting (so big papers don't get extra votes).

In [ ]:
for r in ['As-printed', 'Solution + H900']:
    print(r); display(spearman_per_setting(ph[ph.ht_route == r]))

## 3. Same recipe, different labs: Solution + H900 by paper

In [ ]:
h9 = ph[(ph.ht_route == 'Solution + H900') & ph.YS_MPa.notna()]
order9 = h9.groupby('paper').YS_MPa.median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, p in enumerate(order9):
    s = h9[h9.paper == p]
    ax.scatter(np.full(len(s), i) + np.random.normal(0, .05, len(s)), s.YS_MPa, c=np.where(s.atomisation.astype(str).str.contains('ater'), 'tab:orange', 'tab:blue'))
    ax.hlines(s.YS_MPa.median(), i - .3, i + .3, color='k')
ax.axhline(1170, ls='--', color='k'); ax.set_xticks(range(len(order9))); ax.set_xticklabels(order9, rotation=35, ha='right')
ax.set_ylabel('Yield strength (MPa)'); ax.set_title('Solution + H900 only (orange = water-atomised powder)')
fig.tight_layout(); fig.savefig(FIG / 'h900_by_paper.png', dpi=200)
band = h9[h9.speed_mm_s.between(700, 820)]
print(f'700-820 mm/s: {band.YS_MPa.min():.0f}-{band.YS_MPa.max():.0f} MPa from {band.paper.nunique()} papers')

## 4. Printer or lab? Kruskal–Wallis ε²
Heat-treated rows: each treatment's median is subtracted first, so only the spread *within* a recipe is compared. The printer make is filled down within each paper.

In [ ]:
h = within_treatment(ph)
k = h[h.make.notna()]
rows = {'Heat treated (same treatment)': (k.residual, k)}
ap = ph[(ph.ht_route == 'As-printed') & ph.make.notna()]
rows['As-printed yield'] = (ap.dropna(subset=['YS_MPa']).YS_MPa, ap.dropna(subset=['YS_MPa']))
rows['As-printed density'] = (ap.dropna(subset=['density_pct']).density_pct, ap.dropna(subset=['density_pct']))
out = []
for name, (v, d) in rows.items():
    e_m, p_m, k_m, n = epsilon_squared(v, d.make); e_p, p_p, k_p, _ = epsilon_squared(v, d.paper)
    out.append({'subset': name, 'n': n, 'printers': k_m, 'papers': k_p, 'eps2_printer': e_m, 'p_printer': round(p_m, 3),
                'eps2_paper': e_p, 'p_paper': round(p_p, 3), 'chance_95%_paper': round(shuffle_threshold(v, d.paper), 2)})
pd.DataFrame(out)

## 5. Machine learning: random split vs whole papers held out
Inputs: 4 laser settings + named heat treatment. Baseline = median yield per heat treatment.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
ml = ph[ph.ml_yield].copy()
print(len(ml), 'rows,', ml.paper.nunique(), 'papers')
models = {'Random forest': lambda: RandomForestRegressor(300, min_samples_leaf=3, random_state=0),
          'Linear regression': LinearRegression, 'Baseline (median per treatment)': None}
try:
    from xgboost import XGBRegressor
    models['XGBoost'] = lambda: XGBRegressor(random_state=0, verbosity=0)
except ImportError:
    print('xgboost not installed: pip install xgboost to include it')
res = {name: {'R2 random': cross_validate(ml, f, 'random')[0], 'R2 papers held out': cross_validate(ml, f, 'papers')[0],
              'MAE papers held out (MPa)': cross_validate(ml, f, 'papers')[1]} for name, f in models.items()}
pd.DataFrame(res).T